# Learning the constraint graph from valid behaviour alone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sulcantonin/torchmodal/blob/main/examples/notebooks/03_graph_coloring.ipynb)

This is the paper's strongest quantitative result. We **hide the graph** and show the
model only proper 3-colourings of it. The accessibility relation `A_theta` is then
learned so that the modal colouring axiom

$$ p_c \rightarrow \neg \Diamond p_c $$

("if I have colour c, no world I can access has colour c") holds for every colouring,
while `A` is pushed as large as it can be.

Adjacent pairs never share a colour, so they saturate to 1. Non-adjacent pairs
sometimes do, so the axiom drives them to 0. The constraint graph falls out —
**edge-recovery AUC 1.000** from valid behaviour alone.


In [ ]:
# Colab: install the library. Locally, this is a no-op if it is already present.
try:
    import torchmodal
except ImportError:
    !pip install -q torchmodal
    import torchmodal

print('torchmodal', torchmodal.__version__)


In [ ]:
!pip install -q scikit-learn networkx


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score
from torchmodal import functional as F

N, K, P, SEED = 16, 3, 0.25, 0
torch.manual_seed(SEED)


## A planted 3-colourable graph


In [ ]:
def generate_graph(n, k, p, seed):
    rng = random.Random(seed)
    planted = [rng.randrange(k) for _ in range(n)]
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            if planted[i] != planted[j] and rng.random() < p:
                A[i, j] = A[j, i] = 1.0
    return A, planted

A_true, planted = generate_graph(N, K, P, SEED)
print('nodes:', N, ' edges:', int(A_true.sum() // 2))


## Collect proper colourings — the only thing the model ever sees


In [ ]:
def random_proper_coloring(A, k, rng, tries=500):
    n = A.shape[0]
    for _ in range(tries):
        order = list(range(n)); rng.shuffle(order)
        col = [-1] * n; ok = True
        for v in order:
            used = {col[u] for u in range(n) if A[v, u] and col[u] >= 0}
            free = [c for c in range(k) if c not in used]
            if not free:
                ok = False; break
            col[v] = rng.choice(free)
        if ok:
            return col
    return None

rng = random.Random(SEED + 1)
seen, colorings = {tuple(planted)}, [planted]
while len(colorings) < 60:
    c = random_proper_coloring(A_true, K, rng)
    if c is not None and tuple(c) not in seen:
        seen.add(tuple(c)); colorings.append(c)
print('distinct proper colourings collected:', len(colorings))


## Learn `A_theta` from the axiom


In [ ]:
Ps = [torch.eye(K)[torch.tensor(c)] for c in colorings]
M = len(colorings)
logitsA = nn.Parameter(torch.zeros(N, N))
opt = optim.Adam([logitsA], lr=0.05)
eye = torch.eye(N)
LAM = 2.0

for ep in range(800):
    opt.zero_grad()
    A = torch.sigmoid(logitsA)
    A = 0.5 * (A + A.t()) * (1 - eye)              # symmetric, hollow
    viol = torch.tensor(0.0)
    for Pc in Ps:
        for c in range(K):
            pc = Pc[:, c]
            dia = F.possibility(pc, A, tau=0.1)
            viol = viol + (1.0 - F.implication(pc, F.negation(dia))).clamp(min=0).sum()
    loss = viol / M - LAM * A.mean()               # satisfy the axiom, else grow A
    loss.backward(); opt.step()
    if ep % 200 == 0:
        print(f'epoch {ep:>4}  loss {loss.item():.4f}')


## Did we recover the graph?


In [ ]:
with torch.no_grad():
    A = torch.sigmoid(logitsA)
    A = (0.5 * (A + A.t()) * (1 - eye)).numpy()

iu = np.triu_indices(N, 1)
auc = roc_auc_score(A_true[iu], A[iu])
print(f'edge-recovery AUC (A_theta vs true adjacency) = {auc:.3f}')
print(f'  true edges     : mean A_theta = {A[iu][A_true[iu] == 1].mean():.3f}')
print(f'  true non-edges : mean A_theta = {A[iu][A_true[iu] == 0].mean():.3f}')


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(A_true, cmap='Blues', vmin=0, vmax=1)
ax[0].set_title('true adjacency (hidden from the model)')
im = ax[1].imshow(A, cmap='Blues', vmin=0, vmax=1)
ax[1].set_title(f'learned $A_\\theta$  (AUC {auc:.3f})')
fig.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()


The learned relation reproduces the hidden constraint graph. Nothing but *valid
behaviour* was ever supplied — no edge was shown, and no negative example.
